# 03 · DICOM preprocessing: slices in the right order, orientation and scale

**What this notebook does:** it shows, on the raw DICOM files of one test study, every step performed by
`src/dicom.py` and `src/cache.py`. Training and inference use the same functions; here we check them visually.

**Why it matters:** the old pipeline had four silent bugs. None of them produced an error message; the model
simply learned badly:

| # | trap | old code | new code |
|---|---|---|---|
| 1 | slice order | sorted by file name (file name = random UID) | `ImagePositionPatient` · slice normal |
| 2 | image orientation | none | flips from the direction cosines |
| 3 | physical scale | every image stretched to a fixed pixel size | fixed **160 mm** field of view → 224 px (0.71 mm/px) |
| 4 | right / left knee | none | right knees are mirrored to the left-knee view |
| + | intensity | each slice normalised on its own | one window per **series** |

**Coordinate system (DICOM / LPS):** +x towards the patient's **left**, +y **posterior**, +z towards the **head**.

> To run locally you need DICOMs under `data/test_series/<study>/`
> (one test study can be downloaded with the Kaggle API).

In [ ]:
# Add the project root (the folder that contains src/) to the Python path; works locally and on Kaggle.
import os, sys
from pathlib import Path

def add_project_root():
    cands = [Path.cwd(), Path.cwd().parent]
    if Path("/kaggle/input").exists():
        # os.walk with pruning: never enters the competition DICOM folders (rglob took minutes)
        for d, dirs, files in os.walk("/kaggle/input"):
            dirs[:] = [x for x in dirs if x not in ("train_series", "test_series")]
            if Path(d).name == "src" and "__init__.py" in files:
                cands.append(Path(d).parent)
    for c in cands:
        if (c / "src" / "__init__.py").exists():
            sys.path.insert(0, str(c))
            return c
    raise FileNotFoundError("src/ not found")

PROJECT_ROOT = add_project_root()
print("Project root:", PROJECT_ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pydicom
from scipy.stats import spearmanr

from src.config import DATA_DIR
from src.dicom import analyze_series, load_series_volume, resolve_laterality, _read_pixels
from src.cache import CacheConfig, process_study, SLOT_NAMES

series_df = pd.read_csv(DATA_DIR / "test_series.csv")
local = [s for s in series_df.StudyInstanceUID.unique() if (DATA_DIR / "test_series" / s).exists()]
STUDY = local[0]
study_rows = series_df[series_df.StudyInstanceUID == STUDY].reset_index(drop=True)
print("Study:", STUDY)
study_rows

## 1. Slice order: file names do not give the anatomical order

For every file of every series we compare two orders:
* **file-name order** (old code: `sorted(glob("*.dcm"))`)
* **physical position**: the value of `ImagePositionPatient` on the dominant axis of the slice normal
  (sagittal → x, coronal → y, axial → z).

A Spearman correlation of 1 would mean the file-name order is correct.

In [ ]:
geoms, rows = {}, []
for _, r in study_rows.iterrows():
    d = DATA_DIR / "test_series" / STUDY / r.SeriesInstanceUID
    g = analyze_series(d)
    geoms[r.SeriesInstanceUID] = g
    by_name = sorted(g.headers, key=lambda h: h.path.name)
    pos_name = [h.ipp[g.normal_axis] for h in by_name]
    inst = [h.instance for h in by_name]
    ds = pydicom.dcmread(str(g.headers[0].path), stop_before_pixels=True)
    rows.append({
        "CSV plane": r.Anatomical_Plane, "geometry plane": g.plane, "FS": r.Fluid_Sensitive,
        "slices": len(g.headers),
        "ρ(file name, position)": spearmanr(range(len(pos_name)), pos_name)[0],
        "ρ(InstanceNumber, konum)": spearmanr(inst, pos_name)[0],
        "piksel mm": round(float(g.headers[0].pixel_spacing[0]), 3),
        "matris": f"{g.headers[0].rows}x{g.headers[0].cols}",
        "FOV mm": round(float(g.headers[0].pixel_spacing[0] * g.headers[0].rows), 1),
        "description": getattr(ds, "SeriesDescription", ""),
    })
pd.DataFrame(rows).round(2)

**Reading:**
* `ρ(file name, position)` is close to zero → the old code read the slices in **shuffled** order;
  "three adjacent slices = a 2.5D window" meant three unrelated slices.
* `ρ(InstanceNumber, position)` is ±1: the order is right but the **direction** can be reversed by the
  scanner (−1), so we use the geometry, which does not depend on the scanner.
* Pixel size and FOV vary between series → a fixed mm scale is needed (section 3).

Below: three "adjacent" slices from the middle of the same sagittal series, by file-name order (top) and by
geometric order (bottom).

In [ ]:
sag_id = next(k for k, g in geoms.items() if g.plane == "Sagittal")
g = geoms[sag_id]
by_name = sorted(g.headers, key=lambda h: h.path.name)
m = len(g.headers) // 2
fig, axes = plt.subplots(2, 3, figsize=(10, 7))
for row, (title, seq) in enumerate([("file-name order (OLD)", by_name), ("geometric order (NEW)", g.headers)]):
    for j, h in enumerate(seq[m - 1:m + 2]):
        axes[row, j].imshow(_read_pixels(h), cmap="gray")
        axes[row, j].set_title(f"{title}\nkonum x={h.ipp[0]:.1f} mm", fontsize=9)
        axes[row, j].axis("off")
plt.tight_layout(); plt.show()

## 2. Image orientation

For every series we look at the dominant axis of the row (`r`) and column (`c`) direction cosines and bring the
image to one standard: row direction +x or +y, column direction −z (head up) or +y. Series that a scanner
stored "upside down" then look the same as the others.

In [ ]:
pd.DataFrame([{
    "plane": g.plane, "row axis": "xyz"[g.row_axis], "column axis": "xyz"[g.col_axis],
    "normal axis": "xyz"[g.normal_axis], "flip horizontal": g.flip_lr, "flip vertical": g.flip_ud,
    "Laterality etiketi": g.laterality_tag, "merkez x (mm)": round(g.center_x_mm, 1),
} for g in geoms.values()])

## 3. Right or left?

Medial and lateral findings (both menisci, medial/lateral OA, MCL) swap sides in the image with the knee side.
The model cannot learn this because it does not know which knee it sees. Solution: bring every knee to one
side (left).

1. The DICOM `Laterality` tag first; about half of the studies in this data set do not have it.
2. Otherwise **geometry**: the patient x coordinate of the image centre. `x < 0` → on the patient's right →
   **right knee**. Too close to the midline (|x| < 10 mm) gives no decision.

Mirroring: on coronal / axial images the left-right axis is the image's horizontal axis → horizontal flip;
on sagittal images it is the slice axis → reverse the slice order.

In [ ]:
lat = resolve_laterality(list(geoms.values()))
lat

## 4. Raw image → canonical image

Canonical: ordered, standard orientation, 160 mm FOV → 224×224, mirrored if it is a right knee, and a
series-level 0.5–99.5 percentile window to 0–255 (uint8).

In [ ]:
cfg = CacheConfig()
mirror = lat["side"] == "R"
fig, axes = plt.subplots(2, len(geoms), figsize=(3.2 * len(geoms), 6.5))
for j, (sid, g) in enumerate(geoms.items()):
    raw = _read_pixels(g.headers[len(g.headers) // 2])
    vol = load_series_volume(g, cfg.n_slices, cfg.size, cfg.fov_mm, mirror_to_left=mirror)
    axes[0, j].imshow(raw, cmap="gray"); axes[0, j].set_title(f"HAM {g.plane}\n{raw.shape[0]}x{raw.shape[1]}", fontsize=9)
    axes[1, j].imshow(vol[len(vol) // 2], cmap="gray"); axes[1, j].set_title(f"CANONICAL\n{vol.shape}", fontsize=9)
    for a in axes[:, j]: a.axis("off")
plt.tight_layout(); plt.show()

## 5. Study → slots (what is written to the cache)

`process_study` places the series of a study into 6 slots (plane × fluid-sensitive or not) and takes 16 evenly
spaced slices from the central 80% of each stack. This is exactly what the cache stores:
`uint8 [slot, 16, 224, 224]`.

In [ ]:
res = process_study(STUDY, study_rows, DATA_DIR, "test", cfg)
print(res["meta"])
vols, slots = res["volumes"], res["slots"]
fig, axes = plt.subplots(len(slots), 6, figsize=(13, 2.3 * len(slots)))
for i, (v, s) in enumerate(zip(vols, slots)):
    for j, k in enumerate(np.linspace(0, len(v) - 1, 6).astype(int)):
        axes[i, j].imshow(v[k], cmap="gray"); axes[i, j].axis("off")
        axes[i, j].set_title(f"{SLOT_NAMES[s]} · kesit {k}", fontsize=8)
plt.tight_layout(); plt.show()

## 6. 2.5D window: what the model sees

Three adjacent slices (now truly adjacent) are fed as the three RGB channels. Below: the channels of one window
and their RGB combination; the colour fringes at edges show the difference between neighbouring slices
(information along the third dimension).

In [ ]:
v = vols[0].astype(np.float32) / 255
c = len(v) // 2
win = np.stack([v[c - 1], v[c], v[c + 1]], -1)
fig, axes = plt.subplots(1, 4, figsize=(13, 3.4))
for j, name in enumerate(["kesit i-1 (R)", "kesit i (G)", "kesit i+1 (B)"]):
    axes[j].imshow(win[..., j], cmap="gray"); axes[j].set_title(name); axes[j].axis("off")
axes[3].imshow(win); axes[3].set_title("RGB 2.5D pencere"); axes[3].axis("off")
plt.tight_layout(); plt.show()

## Summary

* Slice order comes from the geometry, orientation from the direction cosines and scale from millimetres.
* Right knees are mirrored; this study had no laterality tag, so the side came from the geometry.
* A broken or missing file does not drop the whole series; errors are written to `meta["errors"]`.
* The same `process_study` function is used for the cache (04) and for test inference (06), so training and
  prediction pixels go through exactly the same path.

**Next:** `04_cache_kaggle.ipynb`, process every training study once on Kaggle and store it.